In [ ]:
import pandas as pd
import numpy as np


In [ ]:
customer_df = pd.read_csv("../data/raw/customer_data.csv")
transactions_df = pd.read_csv("../data/raw/transactions_data.csv")

transactions_df["date"] = pd.to_datetime(transactions_df["date"])

transactions_clean = transactions_df.drop_duplicates().copy()


In [ ]:
print("Customer dataset:", customer_df.shape)
print("Original transactions:", transactions_df.shape)
print("Cleaned transactions:", transactions_clean.shape)
print("Duplicate transactions removed:", len(transactions_df) - len(transactions_clean))
print("Transaction date range:", transactions_clean["date"].min(), "to", transactions_clean["date"].max())

In [ ]:
snapshot_dates = pd.to_datetime([
    "2023-04-30",
    "2023-05-31",
    "2023-06-30",
    "2023-07-31",
    "2023-08-31",
    "2023-09-30",
    "2023-10-31",
    "2023-11-29"
])

history_days = 90
future_days = 30
min_history_transactions = 2

In [ ]:
def create_snapshot_target(
    transactions,
    snapshot_date,
    history_days=90,
    future_days=30,
    min_history_transactions=2
):
    history_start = snapshot_date - pd.Timedelta(days=history_days)
    future_end = snapshot_date + pd.Timedelta(days=future_days)

    historical_transactions = transactions[
        (transactions["date"] > history_start) &
        (transactions["date"] <= snapshot_date)
    ]

    future_transactions = transactions[
        (transactions["date"] > snapshot_date) &
        (transactions["date"] <= future_end)
    ]

    historical_tx_count = (
        historical_transactions
        .groupby("customer_id")
        .size()
        .reset_index(name="historical_tx_count")
    )

    eligible_customers = historical_tx_count[
        historical_tx_count["historical_tx_count"] >= min_history_transactions
    ].copy()

    future_tx_count = (
        future_transactions
        .groupby("customer_id")
        .size()
        .reset_index(name="future_tx_count")
    )

    snapshot_target = eligible_customers.merge(
        future_tx_count,
        on="customer_id",
        how="left"
    )

    snapshot_target["future_tx_count"] = (
        snapshot_target["future_tx_count"]
        .fillna(0)
        .astype(int)
    )

    snapshot_target["churn"] = (
        snapshot_target["future_tx_count"] == 0
    ).astype(int)

    snapshot_target["snapshot_date"] = snapshot_date

    return snapshot_target

In [ ]:
snapshot_targets = pd.concat(
    [
        create_snapshot_target(
            transactions_clean,
            snapshot_date,
            history_days,
            future_days,
            min_history_transactions
        )
        for snapshot_date in snapshot_dates
    ],
    ignore_index=True
)

print("Target dataset shape:", snapshot_targets.shape)
print("Number of snapshots:", snapshot_targets["snapshot_date"].nunique())

snapshot_summary = (
    snapshot_targets
    .groupby("snapshot_date")
    .agg(
        eligible_customers=("customer_id", "nunique"),
        churned_customers=("churn", "sum"),
        churn_rate=("churn", "mean")
    )
)

snapshot_summary["churn_rate"] = (
    snapshot_summary["churn_rate"] * 100
).round(2)

display(snapshot_summary)

In [ ]:
invalid_targets = (
    ((snapshot_targets["churn"] == 1) & (snapshot_targets["future_tx_count"] != 0)) |
    ((snapshot_targets["churn"] == 0) & (snapshot_targets["future_tx_count"] == 0))
).sum()

print("Invalid target rows:", invalid_targets)

print("\nTarget distribution:")
print(snapshot_targets["churn"].value_counts())

print("\nTarget percentages:")
print(
    snapshot_targets["churn"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nDuplicate customer-snapshot combinations:",
      snapshot_targets.duplicated(["customer_id", "snapshot_date"]).sum())

snapshot_targets.to_csv(
    "../data/processed/churn_snapshot_targets.csv",
    index=False
)

In [ ]:
feature_frames = []

transaction_types = [
    "deposit",
    "payment",
    "transfer",
    "withdrawal"
]

for snapshot_date in snapshot_dates:

    history_start = snapshot_date - pd.Timedelta(days=history_days)

    historical_transactions = transactions_clean[
        (transactions_clean["date"] > history_start) &
        (transactions_clean["date"] <= snapshot_date)
    ].copy()

    historical_transactions["is_weekend"] = (
        historical_transactions["date"].dt.dayofweek >= 5
    )

    transaction_features = (
        historical_transactions
        .groupby("customer_id")
        .agg(
            tx_count_90d=("amount", "count"),
            total_amount_90d=("amount", "sum"),
            avg_amount_90d=("amount", "mean"),
            median_amount_90d=("amount", "median"),
            std_amount_90d=("amount", "std"),
            min_amount_90d=("amount", "min"),
            max_amount_90d=("amount", "max"),
            active_days_90d=("date", "nunique"),
            first_tx_date=("date", "min"),
            last_tx_date=("date", "max"),
            weekend_tx_count=("is_weekend", "sum")
        )
        .reset_index()
    )

    transaction_features["days_since_last_tx"] = (
        snapshot_date - transaction_features["last_tx_date"]
    ).dt.days

    transaction_features["days_since_first_tx"] = (
        snapshot_date - transaction_features["first_tx_date"]
    ).dt.days

    transaction_features["weekend_tx_ratio"] = (
        transaction_features["weekend_tx_count"] /
        transaction_features["tx_count_90d"]
    )

    transaction_features["avg_tx_per_active_day"] = (
        transaction_features["tx_count_90d"] /
        transaction_features["active_days_90d"]
    )

    transaction_features = transaction_features.drop(
        columns=["first_tx_date", "last_tx_date"]
    )

    historical_transactions["type"] = (
        historical_transactions["type"].astype(str).str.strip().str.lower()
    )

    type_counts = pd.crosstab(
        historical_transactions["customer_id"],
        historical_transactions["type"]
    )

    type_counts = type_counts.reindex(
        columns=transaction_types,
        fill_value=0
    )

    type_counts.columns = [
        f"tx_type_{col}_count" for col in type_counts.columns
    ]

    type_shares = type_counts.div(
        type_counts.sum(axis=1),
        axis=0
    )

    type_shares.columns = [
        col.replace("_count", "_share")
        for col in type_counts.columns
    ]

    type_features = pd.concat(
        [type_counts, type_shares],
        axis=1
    ).reset_index()

    target_slice = snapshot_targets[
        snapshot_targets["snapshot_date"] == snapshot_date
    ][
        ["customer_id", "snapshot_date", "churn", "historical_tx_count"]
    ]

    snapshot_features = target_slice.merge(
        transaction_features,
        on="customer_id",
        how="left",
        validate="one_to_one"
    )

    snapshot_features = snapshot_features.merge(
        type_features,
        on="customer_id",
        how="left",
        validate="one_to_one"
    )

    feature_frames.append(snapshot_features)

behavioral_features = pd.concat(
    feature_frames,
    ignore_index=True
)

type_feature_columns = [
    f"tx_type_{tx_type}_{suffix}"
    for tx_type in transaction_types
    for suffix in ["count", "share"]
]

behavioral_features[type_feature_columns] = (
    behavioral_features[type_feature_columns].fillna(0)
)

print("Behavioural features shape:", behavioral_features.shape)

In [ ]:
print("Transaction count mismatches:",
      (behavioral_features["tx_count_90d"] != behavioral_features["historical_tx_count"]).sum())

print("Duplicate customer-snapshot combinations:",
      behavioral_features.duplicated(["customer_id", "snapshot_date"]).sum())

print("\nMissing values:")
print(
    behavioral_features.isnull()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\nSnapshot coverage:")
print(behavioral_features["snapshot_date"].value_counts().sort_index())

behavioral_features.to_csv(
    "../data/processed/churn_behavioral_features.csv",
    index=False
)

In [ ]:
profile_columns = [
    "age",
    "gender",
    "location",
    "income_bracket",
    "occupation",
    "education_level",
    "marital_status",
    "household_size",
    "acquisition_channel"
]

customer_profile = customer_df[
    ["customer_id"] + profile_columns
].copy()

print("Customer profile shape:", customer_profile.shape)
print("Duplicate customer IDs:", customer_profile["customer_id"].duplicated().sum())

In [ ]:
final_features = behavioral_features.merge(
    customer_profile,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Merge results:")
print(final_features["_merge"].value_counts())

final_features = final_features.drop(columns="_merge")

print("\nFinal feature dataset shape:", final_features.shape)
print("Total missing values:", final_features.isnull().sum().sum())
print("Duplicate rows:", final_features.duplicated().sum())

print("\nMissing values by column:")
print(
    final_features.isnull()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

In [ ]:
behavioral_columns = [
    "tx_count_90d",
    "total_amount_90d",
    "avg_amount_90d",
    "median_amount_90d",
    "std_amount_90d",
    "min_amount_90d",
    "max_amount_90d",
    "active_days_90d",
    "days_since_last_tx",
    "days_since_first_tx",
    "weekend_tx_ratio",
    "avg_tx_per_active_day",
    "tx_type_deposit_count",
    "tx_type_payment_count",
    "tx_type_transfer_count",
    "tx_type_withdrawal_count",
    "tx_type_deposit_share",
    "tx_type_payment_share",
    "tx_type_transfer_share",
    "tx_type_withdrawal_share"
]

demographic_columns = [
    "age",
    "gender",
    "location",
    "income_bracket",
    "occupation",
    "education_level",
    "marital_status",
    "household_size",
    "acquisition_channel"
]

target_columns = [
    "customer_id",
    "snapshot_date",
    "churn"
]

selected_columns = (
    target_columns +
    behavioral_columns +
    demographic_columns
)

modeling_features = final_features[selected_columns].copy()

print("Final modelling dataset shape:", modeling_features.shape)
print("Number of predictors:", len(behavioral_columns) + len(demographic_columns))

print("\nMissing values:")
print(modeling_features.isnull().sum())

print("\nDuplicate customer-snapshot combinations:",
      modeling_features.duplicated(["customer_id", "snapshot_date"]).sum())

print("\nTarget distribution:")
print(modeling_features["churn"].value_counts())

print("\nTarget percentages:")
print(
    modeling_features["churn"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
modeling_features.to_csv(
    "../data/processed/churn_selected_features.csv",
    index=False
)

print("Final modelling dataset saved successfully.")
print("Dataset shape:", modeling_features.shape)

In [ ]:
train_df = modeling_features[
    modeling_features["snapshot_date"] <= "2023-09-30"
].copy()

validation_df = modeling_features[
    modeling_features["snapshot_date"] == "2023-10-31"
].copy()

test_df = modeling_features[
    modeling_features["snapshot_date"] == "2023-11-29"
].copy()

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Testing shape:", test_df.shape)

print("\nTraining snapshots:")
print(sorted(train_df["snapshot_date"].unique()))

print("\nValidation snapshots:")
print(sorted(validation_df["snapshot_date"].unique()))

print("\nTesting snapshots:")
print(sorted(test_df["snapshot_date"].unique()))

In [ ]:
train_df.to_csv(
    "../data/processed/train_data.csv",
    index=False
)

validation_df.to_csv(
    "../data/processed/validation_data.csv",
    index=False
)

test_df.to_csv(
    "../data/processed/test_data.csv",
    index=False
)

print("Training, validation and testing datasets saved successfully.")